# Explore one ClinicalTrials.gov study in FHIR format

Fetches `GET /studies/{nctId}?format=fhir.json` from the ClinicalTrials.gov API v2 using the helpers in `fetch_study_fhir.py`, then walks the returned FHIR Bundle.

Requires `requests`. Change `NCT_ID` below and run all cells.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import requests

from fetch_study_fhir import check_api_version, fetch_study_fhir, summarize_bundle, validate_nct_id

NCT_ID = validate_nct_id("NCT04280705")

## Fetch

`check_api_version` raises if the live API isn't `2.0.5`. `fetch_study_fhir` follows alias redirects and raises with the API's plain-text message on 400/404.

In [ ]:
with requests.Session() as session:
    print("API version:", check_api_version(session))
    bundle = fetch_study_fhir(session, NCT_ID)

print(json.dumps(summarize_bundle(bundle), indent=2))

In [ ]:
resources = [entry.get("resource", {}) for entry in bundle.get("entry", [])]
study = next((r for r in resources if r.get("resourceType") == "ResearchStudy"), {})


def concept_text(concept):
    """Readable label for a FHIR CodeableConcept: its text, else the first coding display."""
    if not concept:
        return None
    if concept.get("text"):
        return concept["text"]
    return next((c.get("display") for c in concept.get("coding", []) if c.get("display")), None)


Counter(r.get("resourceType") for r in resources)

## Study overview (ResearchStudy)

In [ ]:
overview = {
    "identifiers": [i.get("value") for i in study.get("identifier", [])],
    "title": study.get("title"),
    "other_titles": [l.get("value") for l in study.get("label", [])],
    "status": study.get("status"),
    "phase": concept_text(study.get("phase")),
    "conditions": [concept_text(c) for c in study.get("condition", [])],
    "keywords": [concept_text(k) for k in study.get("keyword", [])],
    "enrollment_actual": study.get("recruitment", {}).get("actualNumber"),
    "enrollment_target": study.get("recruitment", {}).get("targetNumber"),
}
print(json.dumps(overview, indent=2))

In [ ]:
# Status history: recruitment status plus registration milestones, with dates where present
for progress in study.get("progressStatus", []):
    period = progress.get("period", {})
    print(f"{concept_text(progress.get('state')):<40} {period.get('start', '')} {period.get('end', '')}")

## Arms and outcomes

In [ ]:
for group in study.get("comparisonGroup", []):
    print(f"- {group.get('name')} [{concept_text(group.get('type'))}]")
    print(f"    {(group.get('description') or '')[:200]}")

In [ ]:
for objective in study.get("objective", []):
    for measure in objective.get("outcomeMeasure", []):
        print(f"[{concept_text(objective.get('type'))}] {measure.get('name')}")

## Results sections (Composition)

Studies with posted results carry Composition resources for participant flow, baseline, outcome measures, and adverse events. Studies without results have none.

In [ ]:
for composition in (r for r in resources if r.get("resourceType") == "Composition"):
    print(composition.get("title"))
    for section in composition.get("section", []):
        print(f"  - {section.get('title')} ({len(section.get('section', []))} subsections)")

## Save the raw Bundle

`raw/` is git-ignored, so the dump won't be committed.

In [ ]:
out_path = Path("raw") / f"{NCT_ID.upper()}.fhir.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes)")